## ts analysis using access

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import os
import xarray as xr
import xcdat as xc
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm as BM
import pandas as pd
import matplotlib as mpl
import matplotlib.ticker as mticker
import netCDF4
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

In [3]:
from scipy import stats

In [4]:
mpl.rcParams['font.family'] = 'Droid Sans'
mpl.rcParams['font.size'] = 12
# Edit axes parameters
mpl.rcParams['axes.linewidth'] = 1.5
# Tick properties
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.minor.size'] = 3
mpl.rcParams['xtick.major.width'] = 1
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.minor.size'] = 3
mpl.rcParams['ytick.major.width'] = 1
mpl.rcParams['ytick.direction'] = 'out'

In [5]:
### Functions needed for the analysis

In [6]:
import matplotlib as m
from matplotlib.colors import BoundaryNorm as BM
import matplotlib.patches as mpatches

def plot_background(ax):
    ax.add_feature(cfeature.COASTLINE, alpha=0.9, lw=1.1)
    # ax.set_global()
    # ax.add_feature(cfeature.LAND, color='lightgray')
    # ax.add_feature(cfeature.OCEAN, color='lightgray')
    gl = ax.gridlines(draw_labels=True,
                      linewidth=1, color='gray', alpha=0.01, linestyle='--')
    gl.top_labels = False
    # gl.left_labels = False
    # gl.bottom_labels = False
    gl.right_labels = False
    gl.xlines = False
    # gl.xlocator = mticker.FixedLocator([-180, -45, 0, 45, 180])
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'k'}
    gl.ylabel_style = {'size': 10, 'color': 'k'}
    return ax


def plot_maps(x, y, z, titles, labels, cmap, levels, cbar_label = 'Precip', pval = [], nrows=1, ncols=3, figsize=(12,4), land_mask_list = [0], add_patch=False, cbar_orientation='vertical', hatch_type = 'insig'):
    fig, axarr = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, constrained_layout=True, subplot_kw={'projection':ccrs.Robinson(central_longitude=180)})
    
    axlist = axarr.flatten()
    
    for ax in axlist:
        plot_background(ax)
    
    for i in range(len(z)):
        axlist[i].contourf(x, y, z[i], cmap = cmap, transform = ccrs.PlateCarree(central_longitude=0), levels=levels, extend='both')
        axlist[i].set_title(titles[i])
        if i in land_mask_list:
            axlist[i].add_feature(cfeature.LAND, color = 'k', zorder=1)
        if pval != []:
            if hatch_type == 'insig':
                pval_plot = np.ma.masked_less_equal(pval[i], 0.05)
            elif hatch_type == 'sig':
                pval_plot = np.ma.masked_greater(pval[i], 0.05)
            axlist[i].pcolor(x, y, pval_plot, alpha = 0., hatch='////', transform = ccrs.PlateCarree(central_longitude=0))
        axlist[i].set_title(titles[i], fontdict={'fontsize':12})
        axlist[i].text(0.1, 1.05, labels[i], size=16, fontweight='bold', transform=axlist[i].transAxes)
        if add_patch:
            axlist[i].add_patch(mpatches.Rectangle(xy=[120, -65], width=170, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[190, -5], width=80, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[140, -5], width=30, height=10,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
            axlist[i].add_patch(mpatches.Rectangle(xy=[250, -30], width=40, height=20,
                                            facecolor='none', edgecolor='k',
                                            transform=ccrs.PlateCarree()))
        
    norm = BM(levels, 256, extend='both')
    fig.colorbar(m.cm.ScalarMappable(norm = norm, cmap=cmap), ax = axlist, \
                orientation = cbar_orientation, shrink=0.4, aspect = 20, pad = 0.05, label = cbar_label)

In [7]:
from functions import preproc_funcs as funcs

In [8]:
from functions import xr_lowess

In [9]:
import glob

In [12]:
files_pic = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f1/Amon/tauu/gn/files/d20191115/*.nc'))
files_2030 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i1p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2035 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i2p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2040 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i3p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2045 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2050 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i5p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2055 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i6p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2060 = sorted(glob.glob('/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i7p1f2/Amon/tauu/gn/files/d20250307/*.nc'))
files_2045

['/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/tauu_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_010101-020012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/tauu_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_020101-030012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/tauu_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_030101-040012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/tauu_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_040101-050012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files/d20250307/tauu_Amon_ACCESS-ESM1-5_esm-piControl_r1i4p1f2_gn_060101-070012.nc',
 '/g/data/fs38/publications/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/esm-piControl/r1i4p1f2/Amon/tauu/gn/files

In [70]:
ds2030 = xc.open_mfdataset(files_2030)
ds2035 = xc.open_mfdataset(files_2035)
ds2040 = xc.open_mfdataset(files_2040)
ds2045 = xc.open_mfdataset(files_2045)
ds2050 = xc.open_mfdataset(files_2050)
ds2055 = xc.open_mfdataset(files_2055)
ds2060 = xc.open_mfdataset(files_2060)

In [71]:
years = ['2030', '2035', '2040', '2045', '2050', '2055', '2060']

In [72]:
collection = dict(zip(years, [ds2030, ds2035, ds2040, ds2045, ds2050, ds2055, ds2060]))

In [88]:
# Function to process a single model and return the detrended NINO3.4 and precip anomalies
def process_model(model_identifier):
    try:
        # print(f"Processing model: {model_identifier}")
        # Load datasetos
        ds_stable = collection[model_identifier]
        # add custom time ranges
        stable_start_year = int(model_identifier)
        stable_end_year = int(stable_start_year) + 1000
        # var = ds_stable['fld_s16i222'].resample(time='AS-JUN').mean('time').load()
        var = ds_stable.tauu.resample(time='AS-JUN').mean('time').load()
        var['time'] = xr.cftime_range(f'{stable_start_year}-01-01', f'{stable_end_year+1}-01-01', freq='1Y')
        #
        # precip = ds_precip['pr'] * 86400  # Convert kg/m²/s to mm/day

        # Calculate 3d values
        # var_anom = funcs.calc_anom(var, hist_access_r10.sel(time = slice('1960', '1990'))).resample(time = 'AS-JUN').mean('time').load()
        # tos_trend = funcs.calc_trend3d(tos_anom.sel(time = slice('1980', '2014')), 'time')
        # tos_trend_pval = funcs.calc_trend_pval3d(tos_anom.sel(time = slice('1980', '2014')), 'time')

        # calc timeseries values
        # weights = np.cos(np.deg2rad(tos.lat))
        # gmst_anom = tos_anom.weighted(weights).mean(('lat', 'lon'))
        # nino34_index = funcs.detrend1d_check(tos_anom.sel(lat = slice(-5,5), lon = slice(-170+360, -120+360)).weighted(weights).mean(('lat', 'lon')), period=15)
        # wp_sst = tos_anom.sel(lat = slice(-5,5), lon = slice(140, 170)).weighted(weights).mean(('lat', 'lon'))
        # ct_sst = tos_anom.sel(lat = slice(-5,5), lon = slice(190, 270)).weighted(weights).mean(('lat', 'lon'))
        # so_sst = tos_anom.sel(lat = slice(-65, -45), lon = slice(120, 290)).weighted(weights).mean(('lat', 'lon'))

        # print(f'Completed: {model_identifie}')
        return model_identifier, var
    except Exception as e:
        print(f"Error processing {model_identifier}: {e}")
        return model_identifier



In [99]:
import xesmf as xe

In [89]:
import multiprocessing as mp

In [100]:
ds_out = xe.util.cf_grid_2d(-0.75, 360, 1.5, -90, 90, 1.5)
ds_out

<xarray.Dataset>
Dimensions:             (lon: 240, bound: 2, lat: 120)
Coordinates:
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
    latitude_longitude  float64 nan
Dimensions without coordinates: bound
Data variables:
    lon_bounds          (lon, bound) float64 -0.75 0.75 0.75 ... 357.8 359.2
    lat_bounds          (lat, bound) float64 -90.0 -88.5 -88.5 ... 88.5 90.0

In [90]:
# Run multiprocessing and gather results
res_arr = []
with mp.Pool(processes=mp.cpu_count()) as pool:
    i = 0
    for res in pool.imap(process_model, years):
        res_arr.append(res)
        print(f'Completed {i+1}/{len(years)}', end='\r')
        i += 1



In [91]:
variants = []
ds_arr = []
for i in range(7):
    variants.append(res_arr[i][0])
    ds_arr.append(res_arr[i][1])

In [93]:
variants

['2030', '2035', '2040', '2045', '2050', '2055', '2060']

In [96]:
ds_arr

[<xarray.DataArray 'tauu' (time: 1001, lat: 145, lon: 192)>
 array([[[-5.97529206e-03, -5.46590611e-03, -4.95066727e-03, ...,
          -7.46291131e-03, -6.97432971e-03, -6.47827890e-03],
         [-1.46298734e-02, -1.39673743e-02, -1.32468361e-02, ...,
          -1.73655916e-02, -1.67871509e-02, -1.42634241e-02],
         [-2.19267495e-02, -2.05638837e-02, -1.90134607e-02, ...,
          -2.43676249e-02, -2.38767304e-02, -2.24746764e-02],
         ...,
         [-3.46090682e-02, -3.46101858e-02, -3.44900303e-02, ...,
          -3.43954600e-02, -3.44714820e-02, -3.45231667e-02],
         [-3.23425010e-02, -3.20384279e-02, -3.17528136e-02, ...,
          -3.32088917e-02, -3.30137163e-02, -3.26434150e-02],
         [-2.35259272e-02, -2.32605189e-02, -2.29701977e-02, ...,
          -2.41700020e-02, -2.39809118e-02, -2.37661432e-02]],
 
        [[-6.18975284e-03, -5.61796268e-03, -5.04015619e-03, ...,
          -7.86294416e-03, -7.31284544e-03, -6.75491663e-03],
         [-1.47446143e-02, 

In [97]:
out = xr.concat([x.to_dataset(name='tauu') for x in ds_arr], dim=variants).rename(dict(concat_dim = 'model'))

In [101]:
regridder = xe.Regridder(out, ds_out, 'bilinear', periodic=True, ignore_degenerate=True)

In [102]:
out_regrid = regridder(out)
out_regrid

<xarray.Dataset>
Dimensions:             (model: 7, time: 1031, lat: 120, lon: 240)
Coordinates:
  * time                (time) object 2030-12-31 00:00:00 ... 3060-12-31 00:0...
  * model               (model) object '2030' '2035' '2040' ... '2055' '2060'
    latitude_longitude  float64 nan
  * lon                 (lon) float64 0.0 1.5 3.0 4.5 ... 355.5 357.0 358.5
  * lat                 (lat) float64 -89.25 -87.75 -86.25 ... 86.25 87.75 89.25
Data variables:
    tauu                (model, time, lat, lon) float32 -0.01116 ... -0.01379
Attributes:
    regrid_method:  bilinear

In [104]:
out_regrid.to_netcdf('/g/data/ob22/as8561/data/enso_trans_stable/sst_grad_res/access_stable_tauu_original.nc')